# Lesson 22: Deflategate and the *t* Distribution

**January 2015, AFC Championship.** The NFL suspected the New England Patriots of letting air out of their footballs. League rules require 12.5 to 13.5 psi, and the Patriots' balls were set at 12.5 psi before the game. Cold air lowers pressure on its own: a physics estimate put the expected halftime pressure at about **11.5 psi** even if nobody touched the balls.

At halftime, an official measured all **11** Patriots footballs.

$$H_0: \mu = 11.5 \quad \text{(the cold explains it)} \qquad H_a: \mu < 11.5 \quad \text{(the balls are lower than the cold explains)}$$

**The question for today:** with only 11 balls, can we still use the $z$ table? Run each cell in order (Shift + Enter).

> Colab setup: *Runtime → Change runtime type → R*.

## 1. The data

We compute the **test statistic**, the same formula as the large-sample test:

$$\frac{\bar{x} - \mu_0}{s/\sqrt{n}}$$

Notice the $s$. We don't know the true spread $\sigma$ of all footballs, so we plug in the spread of our 11.

In [ ]:
library(ggplot2)
set.seed(206)

# Halftime readings of the 11 Patriots footballs (psi)
psi <- c(10.50, 10.70, 10.85, 10.90, 10.95, 11.10, 11.10, 11.15, 11.50, 11.60, 11.85)
mu0 <- 11.5          # what the cold alone should produce
n   <- length(psi)

xbar <- mean(psi)
s    <- sd(psi)
t_stat <- (xbar - mu0) / (s / sqrt(n))
cat("n =", n, "  sample mean =", round(xbar, 3), "  s =", round(s, 3),
    "  test statistic =", round(t_stat, 2), "\n")

# Stack repeated readings so both 11.10 balls show
dots <- data.frame(psi)
dots$height <- ave(dots$psi, dots$psi, FUN = seq_along)

options(repr.plot.width = 10, repr.plot.height = 3.5)
ggplot(dots, aes(psi, height)) +
  geom_point(size = 6, color = "navy", alpha = 0.8) +
  geom_vline(xintercept = mu0, linetype = "dashed", linewidth = 1) +
  annotate("text", x = mu0 - 0.02, y = 2.6, hjust = 1, size = 5,
           label = "11.5 psi: what the cold alone explains") +
  scale_y_continuous(limits = c(0.5, 3), breaks = NULL) +
  scale_x_continuous(breaks = seq(10.4, 12, by = 0.2)) +
  labs(title = "The 11 Patriots footballs at halftime",
       x = "pressure (psi)", y = NULL) +
  theme_minimal(base_size = 15) +
  theme(plot.background = element_rect(fill = "white", color = NA))

## 2. The problem: $s$ is a guess

To see what goes wrong, we build a world where we **know** the Patriots are innocent: every ball comes from a population averaging exactly 11.5 psi with true spread $\sigma = 0.4$. Then we "play" 10,000 games, measure 11 balls each game, and compute $s$ each time.

If $s$ were a reliable stand-in for $\sigma$, every bar would sit near 0.4.

In [ ]:
options(repr.plot.width = 12, repr.plot.height = 7, repr.plot.res = 120)   # big plots for the rest of the notebook

# Pretend the Patriots are innocent: every ball really averages 11.5 psi.
# Each "game" we measure 11 balls and compute s, the sample standard deviation.
sigma <- 0.4        # the true spread (we only know this because we made up the world)
reps  <- 10000

samples <- matrix(rnorm(n * reps, mean = mu0, sd = sigma), nrow = n)  # one column per game
xbars <- colMeans(samples)
sds   <- apply(samples, 2, sd)

sim <- data.frame(xbar = xbars, s = sds)
summary(sim$s)

ggplot(sim, aes(s)) +
  geom_histogram(binwidth = 0.02, fill = "grey70", color = "white", boundary = 0) +
  geom_vline(xintercept = sigma, linewidth = 1.2, color = "firebrick") +
  annotate("text", x = sigma + 0.01, y = Inf, vjust = 1.5, hjust = 0, size = 5,
           color = "firebrick", label = "true sigma = 0.4") +
  labs(title = "With only 11 balls, s is a shaky guess for sigma",
       subtitle = "10,000 honest games: the s we compute lands anywhere from about 0.25 to 0.55",
       x = "sample standard deviation s", y = "number of games") +
  theme_minimal(base_size = 15) +
  theme(plot.background = element_rect(fill = "white", color = NA))

**What to notice:** $s$ swings a lot. In about 1 game out of 7 it comes out under 0.3, well below the truth. With 400 data points, $s$ would almost always land very close to 0.4. With 11, it doesn't.

## 3. What that does to the test statistic

Now compute the test statistic for every honest game. If the $z$ test works, the histogram should follow the red $z$ curve.

In [ ]:
# Compute the test statistic for every honest game, using s (all we ever have in real life)
sim$t_stat <- (sim$xbar - mu0) / (sim$s / sqrt(n))

ggplot(sim, aes(t_stat)) +
  geom_histogram(aes(y = after_stat(density)), binwidth = 0.2,
                 fill = "grey75", color = "white", boundary = 0) +
  geom_function(fun = dnorm, aes(color = "z curve (standard normal)"), linewidth = 1.2) +
  geom_function(fun = dt, args = list(df = n - 1),
                aes(color = "t curve, df = 10"), linewidth = 1.2, linetype = "dashed") +
  scale_color_manual(values = c("z curve (standard normal)" = "firebrick",
                                "t curve, df = 10" = "navy")) +
  coord_cartesian(xlim = c(-5, 5)) +
  labs(title = "The test statistic for 10,000 HONEST games (n = 11)",
       subtitle = "The bars spill past the z curve in both tails. The t curve fits.",
       x = "test statistic = (xbar - 11.5) / (s / sqrt(11))", y = "density", color = NULL) +
  theme_minimal(base_size = 15) +
  theme(legend.position = "top",
        plot.background = element_rect(fill = "white", color = NA))

# Zoom in on the left tail, where decisions get made. Heights are now counts of games.
bw <- 0.1
ggplot(subset(sim, t_stat < -1.5), aes(t_stat)) +
  geom_histogram(binwidth = bw, boundary = -1.5, fill = "grey75", color = "white") +
  geom_function(fun = function(x) dnorm(x) * reps * bw,
                aes(color = "z curve (standard normal)"), linewidth = 1.2) +
  geom_function(fun = function(x) dt(x, n - 1) * reps * bw,
                aes(color = "t curve, df = 10"), linewidth = 1.2, linetype = "dashed") +
  scale_color_manual(values = c("z curve (standard normal)" = "firebrick",
                                "t curve, df = 10" = "navy")) +
  coord_cartesian(xlim = c(-5, -1.5)) +
  labs(title = "Zoomed in on the left tail",
       subtitle = "Past -2.5, honest games keep landing where z says they almost never should",
       x = "test statistic", y = "number of honest games", color = NULL) +
  theme_minimal(base_size = 15) +
  theme(legend.position = "top",
        plot.background = element_rect(fill = "white", color = NA))

**What to notice:** in the middle, $z$ and $t$ look almost the same. The difference lives in the **tails**, which is exactly where we make reject decisions. The dashed blue $t$ curve tracks the bars; the red $z$ curve runs underneath them.

## 4. Why the tails get fat

The test statistic divides by $s$. When $s$ happens to come out too **small**, we divide by something small and the test statistic gets stretched, even though nothing is wrong with the footballs.

In [ ]:
# Why the fat tails? When s happens to come out SMALL, we divide by something small,
# and the test statistic blows up -- even though nothing is wrong with the footballs.
sim$s_size <- ifelse(sim$s < 0.3, "s came out small (< 0.3)", "s came out 0.3 or more")

ggplot(sim, aes(xbar, t_stat, color = s_size)) +
  geom_point(alpha = 0.25, size = 1) +
  geom_hline(yintercept = -1.645, linetype = "dashed") +
  annotate("text", x = min(sim$xbar), y = -1.645, vjust = -0.5, hjust = 0, size = 4.5,
           label = "z cutoff: -1.645") +
  scale_color_manual(values = c("s came out small (< 0.3)" = "firebrick",
                                "s came out 0.3 or more" = "grey60")) +
  labs(title = "Same sample mean, very different test statistic",
       subtitle = "Red games underestimated the spread, so their test statistics get stretched",
       x = "sample mean (psi)", y = "test statistic", color = NULL) +
  guides(color = guide_legend(override.aes = list(alpha = 1, size = 3))) +
  theme_minimal(base_size = 15) +
  theme(legend.position = "top",
        plot.background = element_rect(fill = "white", color = NA))

**What to notice:** the red games (small $s$) fan out away from zero. Many of them cross the $-1.645$ cutoff only because their $s$ was unluckily small. Those are false accusations.

## 5. How often do we accuse an innocent team?

A 5% test should wrongly reject an innocent team 5% of the time. Let's check the $z$ cutoff of $-1.645$ in our honest world, then repeat for different sample sizes.

In [ ]:
# Use the z cutoff (-1.645) to make a 5% test. How often do we accuse an INNOCENT team?
false_alarm <- mean(sim$t_stat < qnorm(0.05))
c(promised = 0.05, actual = round(false_alarm, 3))

# Repeat for different numbers of footballs
false_alarm_rate <- function(n, reps = 10000) {
  x <- matrix(rnorm(n * reps, mu0, sigma), nrow = n)
  r <- (colMeans(x) - mu0) / (apply(x, 2, sd) / sqrt(n))
  mean(r < qnorm(0.05))
}
sizes <- c(3, 5, 8, 11, 20, 40, 100)
fa <- data.frame(n = sizes, rate = sapply(sizes, false_alarm_rate))
fa

ggplot(fa, aes(n, rate)) +
  geom_hline(yintercept = 0.05, linetype = "dashed", linewidth = 1) +
  annotate("text", x = 3, y = 0.05, label = "the 5% the z test promises",
           hjust = 0, vjust = -0.6, size = 4.5) +
  geom_line(linewidth = 1.2, color = "firebrick") +
  geom_point(size = 3, color = "firebrick") +
  scale_x_log10(breaks = sizes) +
  labs(title = "Using the z cutoff with small samples accuses innocent teams too often",
       subtitle = "By about n = 40 the z test is close enough",
       x = "number of footballs measured (n)", y = "false alarm rate") +
  theme_minimal(base_size = 15) +
  theme(plot.background = element_rect(fill = "white", color = NA))

**What to notice:** with 3 footballs, the "5%" test wrongly rejects about 12% of the time, more than double what it promises. As $n$ grows, $s$ settles down near $\sigma$ and the error rate falls to 5%. That's why our rule is $n > 40$ for the $z$ test.

## 6. The fix: the $t$ distribution

Instead of pretending $s = \sigma$, we use a curve that already accounts for the extra wobble in $s$: the **$t$ distribution**. It has one setting, the **degrees of freedom**, $\nu = n - 1$.

In [ ]:
# The t family: fewer data points -> fatter tails. More data -> t becomes z.
x <- seq(-4, 4, by = 0.01)
curves <- rbind(
  data.frame(x, y = dt(x, 2),  curve = "t, df = 2"),
  data.frame(x, y = dt(x, 5),  curve = "t, df = 5"),
  data.frame(x, y = dt(x, 10), curve = "t, df = 10"),
  data.frame(x, y = dt(x, 40), curve = "t, df = 40"),
  data.frame(x, y = dnorm(x),  curve = "z"))
curves$curve <- factor(curves$curve, levels = c("t, df = 2", "t, df = 5", "t, df = 10", "t, df = 40", "z"))

ggplot(curves, aes(x, y, color = curve)) +
  geom_line(linewidth = 1.1) +
  scale_color_manual(values = c("#c6dbef", "#6baed6", "#2171b5", "#08306b", "firebrick")) +
  labs(title = "The t curves close in on z as the sample grows",
       x = NULL, y = "density", color = NULL) +
  theme_minimal(base_size = 15) +
  theme(legend.position = "top",
        plot.background = element_rect(fill = "white", color = NA))

# Cutoffs for a one-sided 5% test
data.frame(df = c(2, 5, 10, 40, Inf), cutoff = round(qt(0.05, c(2, 5, 10, 40, Inf)), 3))

**What to notice:** the fewer data points, the fatter the tails, and the farther out the cutoff. For 11 balls ($\nu = 10$), a 5% one-sided test needs $t < -1.812$, not $-1.645$. By $\nu = 40$, the cutoff is almost back to $z$.

## 7. Back to Deflategate

The Patriots' test statistic was $-3.22$. In our innocent world, how often does a game produce a test statistic that extreme?

In [ ]:
# Back to Deflategate. How often does an honest game produce a test statistic as extreme as -3.22?
t_obs <- t_stat
p_sim <- mean(sim$t_stat <= t_obs)
p_t   <- pt(t_obs, df = n - 1)
p_z   <- pnorm(t_obs)
round(c(simulation = p_sim, t_table = p_t, z_table = p_z), 4)

# Zoom in on the tail: every honest game that was at least as extreme as the Patriots
bw <- 0.1
ggplot(subset(sim, t_stat < -2), aes(t_stat)) +
  geom_histogram(aes(fill = t_stat <= t_obs), binwidth = bw,
                 boundary = t_obs, color = "white") +
  scale_fill_manual(values = c(`TRUE` = "firebrick", `FALSE` = "grey75"), guide = "none") +
  geom_function(fun = function(x) dt(x, n - 1) * reps * bw,
                aes(color = "t curve, df = 10"), linewidth = 1.1) +
  geom_function(fun = function(x) dnorm(x) * reps * bw,
                aes(color = "z curve"), linewidth = 1.1, linetype = "dashed") +
  scale_color_manual(values = c("t curve, df = 10" = "navy", "z curve" = "firebrick")) +
  geom_vline(xintercept = t_obs, linewidth = 1) +
  annotate("text", x = t_obs - 0.1, y = 40, hjust = 1, size = 4.5,
           label = paste0("Patriots: ", round(t_obs, 2),
                          "\nhonest games this extreme: ", sum(sim$t_stat <= t_obs), " of ", reps,
                          "\nsimulation P = ", sprintf("%.4f", p_sim),
                          "\nt table P = ", sprintf("%.4f", p_t),
                          "\nz table P = ", sprintf("%.4f", p_z))) +
  coord_cartesian(xlim = c(-6, -2.15)) +
  labs(title = "How rare is the Patriots' result if they are innocent?",
       subtitle = "Red = honest games at least as extreme. The simulation matches t; z says about 8x rarer.",
       x = "test statistic from honest games", y = "number of honest games", color = NULL) +
  theme_minimal(base_size = 15) +
  theme(legend.position = "top",
        plot.background = element_rect(fill = "white", color = NA))

**What to notice:** the simulation, built only from honest games, gives about the same P-value as the $t$ table. The $z$ table says 0.0006, roughly **eight times** stronger evidence than the data actually support. With a small sample, $z$ makes you overconfident.

**Conclusion (at $\alpha = 0.05$):** P-value $= 0.0046 < 0.05$, so we reject $H_0$. The balls were lower than cold weather alone explains.

## 8. In one line of R, plus a twist

`t.test` does everything above. Then we run it on the **second official's** readings of the same 11 balls, taken with a different gauge.

In [ ]:
# Everything above, in one line
t.test(psi, mu = 11.5, alternative = "less")

# The second official's gauge read the same balls about 0.4 psi higher
psi2 <- c(11.80, 11.20, 11.50, 11.00, 11.45, 11.95, 12.30, 11.55, 11.35, 10.90, 11.35)
t.test(psi2, mu = 11.5, alternative = "less")

**Same balls, opposite conclusion.** The second gauge read about 0.4 psi higher on every ball, and the P-value jumps from 0.005 to 0.46. A test can only be as good as the measurements going into it. Statistics told us how strong the evidence was *given the gauge*. It could not tell us which gauge was right.

### Questions to discuss
1. In Section 5, why does the false alarm rate get worse as $n$ gets smaller?
2. If we had measured 100 footballs, would it matter much whether we used $z$ or $t$? Use the plot in Section 6.
3. Change `sigma <- 0.4` to `sigma <- 2` in Section 2 and rerun Sections 2 through 5. Does the false alarm rate change? Why or why not?